# 01 – Yelp Data Exploration

Quick overview of the Yelp dataset via DuckDB (the same approach as
`src/yelp_initial_exploration/yelp_build_csr.py`).

**Pre-requisites** – convert Yelp JSON files to Parquet first:
see `data/README.md` for instructions.  Then set `PARQUET_DIR` below.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

PARQUET_DIR = Path("../yelp_parquet")  # adjust if needed
business_glob = str(PARQUET_DIR / "business" / "state=*" / "*.parquet")
review_glob  = str(PARQUET_DIR / "review"   / "year=*"  / "*.parquet")

con = duckdb.connect()
print("DuckDB connected")

## 1. Business overview

In [ ]:
biz_stats = con.execute(f"""
    SELECT state, COUNT(*) AS n_businesses,
           ROUND(AVG(review_count), 1) AS avg_reviews
    FROM read_parquet('{business_glob}')
    GROUP BY state
    ORDER BY n_businesses DESC
    LIMIT 15
""").fetchdf()
biz_stats

## 2. Review star distribution

In [ ]:
star_dist = con.execute(f"""
    SELECT TRY_CAST(stars AS INTEGER) AS stars, COUNT(*) AS n
    FROM read_parquet('{review_glob}')
    GROUP BY stars ORDER BY stars
""").fetchdf()

star_dist.plot.bar(x='stars', y='n', legend=False, title='Review star distribution')
plt.xlabel('Stars'); plt.ylabel('Count'); plt.tight_layout(); plt.show()

## 3. Implicit-feedback sparsity (stars ≥ 4)

In [ ]:
pos = con.execute(f"""
    SELECT COUNT(DISTINCT user_id) AS n_users,
           COUNT(DISTINCT business_id) AS n_items,
           COUNT(*) AS n_interactions
    FROM read_parquet('{review_glob}')
    WHERE TRY_CAST(stars AS DOUBLE) >= 4.0
      AND user_id IS NOT NULL
      AND business_id IS NOT NULL
""").fetchdf()

n_u, n_i, n_int = pos.iloc[0]
density = n_int / (n_u * n_i) * 100
print(f"Users: {int(n_u):,}  Items: {int(n_i):,}  Interactions: {int(n_int):,}")
print(f"Matrix density: {density:.4f}%")

## 4. User-interaction distribution (log scale)

In [ ]:
user_counts = con.execute(f"""
    SELECT user_id, COUNT(*) AS n
    FROM read_parquet('{review_glob}')
    WHERE TRY_CAST(stars AS DOUBLE) >= 4.0
    GROUP BY user_id
""").fetchdf()

user_counts['n'].hist(bins=50, log=True)
plt.title('Reviews per user (log y-axis)')
plt.xlabel('Review count'); plt.ylabel('Number of users (log)')
plt.tight_layout(); plt.show()